# AgentLint: train the binary detector

Runtime → Change runtime type → **T4 GPU**. Trains two models on `data/splits/full`
(val only; `test` is never read):

1. `configs/binary.yaml`: fine-tuned GraphCodeBERT (chunk + max-pool, weighted BCE).
2. `configs/binary_frozen.yaml`: the "before fine-tuning" baseline (frozen encoder, head only).

Each run writes `metrics.json`, `val_predictions.jsonl`, `threshold.json` and `best/` to its
`output_dir`. The last cell copies them to Google Drive (Colab's disk is wiped).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "main"
!git clone -b {BRANCH} https://github.com/tomkkl/mismatch-detection.git
%cd mismatch-detection
!pip install -q -e .

## Smoke test

A few batches on a slice of the data, to catch errors before the long run (~1 min).

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/binary.yaml"))
cfg.update(epochs=1, limit_train=64, limit_val=32, output_dir="checkpoints/smoke")
yaml.safe_dump(cfg, open("configs/smoke.yaml", "w"))
!agentlint train configs/smoke.yaml

## 1. Fine-tuned model

In [ ]:
!agentlint train configs/binary.yaml

## 2. Frozen-encoder baseline

In [ ]:
!agentlint train configs/binary_frozen.yaml

## Compare (val)

Best epoch by PR-AUC. `@best-F1` uses the threshold tuned on val. Check the chunk-count
breakdown: honest pairs with 2+ chunks should not be flagged much more often than 1-chunk ones.

In [ ]:
import json
for name in ["binary", "binary_frozen"]:
    r = json.load(open(f"checkpoints/{name}/metrics.json"))["best"]
    for at in ["val_at_0.5", "val_at_best_f1"]:
        m = r[at]
        print(f"{name:14s} {at:15s} epoch {r['epoch']} | P {m['precision']:.3f} R {m['recall']:.3f} "
              f"F1 {m['f1']:.3f} PR-AUC {m['pr_auc']:.3f} group acc {m['group_accuracy']:.3f} "
              f"(t={m['threshold']:.2f})")
    for k, v in r["val_at_best_f1"]["breakdowns"]["chunks"].items():
        print(f"    chunks {k:3s} n {v['n']:3d} mismatches {v['n_mismatch']:3d} | P {v['precision']:.3f} R {v['recall']:.3f}")

## Save to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/agentlint
!rsync -a --exclude smoke checkpoints/ /content/drive/MyDrive/agentlint/checkpoints/